# 01 — EDA e Extração de Features (ICBHI 2017)

**Objetivo:** construir o DataFrame consolidado de gravações respiratórias do dataset ICBHI 2017, explorar a distribuição de classes e demonstrar a extração de features de áudio com `librosa` — base para um assistente de diagnóstico de doenças pulmonares.

**Etapas:**
1. Carregar metadata (diagnósticos, demografia, parsing dos nomes de arquivo)
2. EDA do DataFrame: classes, equipamentos, locais de auscultação, durações
3. Análise dos ciclos respiratórios anotados (crackles / wheezes)
4. Visualização de áudios: waveform, mel-spectrogram, MFCC
5. Demonstração da extração de vetor de features por gravação

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

from src import config, metadata, features

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
print("librosa", librosa.__version__)
print("ICBHI root:", config.ICBHI_ROOT)

## 1. Metadata consolidada

Combinamos três fontes:
- `patient_diagnosis.csv` — rótulo clínico por paciente
- `demographic_info.txt` — idade, sexo, BMI
- nome do `.wav` — paciente, índice de gravação, local do tórax, modo, equipamento

In [ ]:
df = metadata.build_recordings_dataframe()
print(f"Total de gravações: {len(df)} | pacientes únicos: {df['patient_id'].nunique()}")
df.head()

In [ ]:
df.info()
print("\nNulos por coluna:")
df.isna().sum()

## 2. EDA — distribuições

In [ ]:
diag_counts = df["diagnosis"].value_counts()
print(diag_counts)

fig, ax = plt.subplots(figsize=(9, 4))
diag_counts.plot(kind="bar", color="steelblue", ax=ax)
ax.set_title("Gravações por diagnóstico (ICBHI 2017)")
ax.set_ylabel("# gravações")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
patient_diag = df.drop_duplicates("patient_id")[["patient_id", "diagnosis"]]
print("Pacientes por classe:")
print(patient_diag["diagnosis"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["equipment"].value_counts().plot(kind="barh", ax=axes[0], color="teal")
axes[0].set_title("Equipamento de captura")
df["chest_location"].value_counts().plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("Local de auscultação")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["age"].dropna(), bins=20, ax=axes[0], color="slateblue")
axes[0].set_title("Idade dos pacientes")
sns.countplot(x="sex", data=df, ax=axes[1])
axes[1].set_title("Sexo")
plt.tight_layout()
plt.show()

## 3. Definição do alvo — filtro de classes e rótulos

Decisão de modelagem: focar em **3 classes clinicamente coerentes** (Healthy, COPD, URTI) e oferecer também o corte binário (Saudável vs Doente). Asthma (1 amostra), Pneumonia (6 pacientes), Bronchiectasis, Bronchiolitis e LRTI são descartadas por inviabilidade estatística.

Duas colunas-alvo:
- `label_multi`: tarefa multiclasse (3 classes)
- `label_binary`: Healthy vs Disease (COPD ∪ URTI)

⚠️ **Importante:** a contagem por **paciente** importa mais que por gravação para evitar vazamento — todo split (treino/teste, k-fold) deve ser **paciente-estratificado** (`StratifiedGroupKFold` com `groups=patient_id`).

In [ ]:
df_target = metadata.filter_target_classes(df)
df_target = metadata.add_target_labels(df_target)

print(f"Gravações: {len(df_target)} | Pacientes: {df_target['patient_id'].nunique()}")
df_target[["filename", "patient_id", "diagnosis", "label_multi", "label_binary"]].head()

In [ ]:
def class_summary(d, col):
    return pd.DataFrame({
        "gravações": d[col].value_counts(),
        "pacientes": d.drop_duplicates("patient_id")[col].value_counts(),
    }).fillna(0).astype(int)

print("--- Multiclasse (3) ---")
print(class_summary(df_target, "label_multi"))
print("\n--- Binário ---")
print(class_summary(df_target, "label_binary"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
class_summary(df_target, "label_multi").plot(kind="bar", ax=axes[0])
axes[0].set_title("Multiclasse: gravações vs pacientes")
axes[0].set_xlabel(""); axes[0].tick_params(axis="x", rotation=0)
class_summary(df_target, "label_binary").plot(kind="bar", ax=axes[1], color=["steelblue", "indianred"])
axes[1].set_title("Binário: gravações vs pacientes")
axes[1].set_xlabel(""); axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

## 4. Ciclos respiratórios anotados

Cada `.txt` lista os ciclos da gravação com flags `crackles` e `wheezes` — esses são os achados auscultaórios clássicos usados pelo médico. Trabalhamos aqui apenas com as gravações filtradas (`df_target`).

In [ ]:
cycles = metadata.build_cycles_dataframe(df_target)
print(f"Total de ciclos: {len(cycles)}")
cycles.head()

In [ ]:
labels = cycles.apply(
    lambda r: (
        "normal" if r.crackles == 0 and r.wheezes == 0
        else "crackles" if r.crackles == 1 and r.wheezes == 0
        else "wheezes" if r.crackles == 0 and r.wheezes == 1
        else "both"
    ),
    axis=1,
)
print(labels.value_counts())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
labels.value_counts().plot(kind="bar", ax=axes[0], color="darkseagreen")
axes[0].set_title("Ciclos por achado auscultaório")
sns.histplot(cycles["duration"], bins=40, ax=axes[1], color="indianred")
axes[1].set_title("Duração dos ciclos (s)")
plt.tight_layout()
plt.show()

## 5. Visualização de um áudio com `librosa`

Comparação de um paciente **Healthy** com um paciente **COPD** — waveform, mel-spectrograma e MFCCs.

In [ ]:
sample_healthy = df_target[df_target["label_multi"] == "Healthy"].iloc[0]
sample_copd = df_target[df_target["label_multi"] == "COPD"].iloc[0]
print("Healthy:", sample_healthy["filename"])
print("COPD   :", sample_copd["filename"])

In [ ]:
def plot_audio(path, title):
    y, sr = features.load_audio(path, duration=10.0)
    mel_db = features.extract_mel_spectrogram(y, sr)
    mfcc = features.extract_mfcc(y, sr)

    fig, axes = plt.subplots(3, 1, figsize=(12, 8))
    fig.suptitle(title, fontsize=13)

    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title("Waveform")

    img1 = librosa.display.specshow(mel_db, sr=sr, hop_length=config.HOP_LENGTH,
                                     x_axis="time", y_axis="mel", ax=axes[1])
    axes[1].set_title("Mel-spectrogram (dB)")
    fig.colorbar(img1, ax=axes[1], format="%+2.0f dB")

    img2 = librosa.display.specshow(mfcc, sr=sr, hop_length=config.HOP_LENGTH,
                                     x_axis="time", ax=axes[2])
    axes[2].set_title(f"MFCC ({config.N_MFCC} coef.)")
    fig.colorbar(img2, ax=axes[2])

    plt.tight_layout()
    plt.show()

plot_audio(sample_healthy["wav_path"], f"Healthy — {sample_healthy['filename']}")
plot_audio(sample_copd["wav_path"], f"COPD — {sample_copd['filename']}")

## 6. Extração de vetor de features

`features.extract_from_file` retorna ~78 features agregadas (MFCC mean/std + deltas, chroma, contrast, ZCR, RMS, centroide, rolloff, bandwidth, flatness, duração). Pronto para alimentar modelos clássicos (RandomForest, XGBoost, SVM).

Aqui rodamos sobre uma **amostra estratificada de 20 gravações** só para validar o pipeline. A extração completa (888 áudios) ficará para o próximo notebook.

In [ ]:
from tqdm.auto import tqdm

sample = (
    df_target.groupby("label_multi", group_keys=False)
    .apply(lambda g: g.sample(min(5, len(g)), random_state=42))
    .reset_index(drop=True)
)
print("Amostra estratificada:")
print(sample["label_multi"].value_counts())

rows = []
for _, r in tqdm(sample.iterrows(), total=len(sample)):
    feats = features.extract_from_file(r["wav_path"])
    feats["filename"] = r["filename"]
    feats["patient_id"] = r["patient_id"]
    feats["label_multi"] = r["label_multi"]
    feats["label_binary"] = r["label_binary"]
    rows.append(feats)

features_df = pd.DataFrame(rows)
print("\nshape:", features_df.shape)
features_df.head()

In [ ]:
out = config.PROCESSED_DIR / "features_sample20.csv"
features_df.to_csv(out, index=False)
print("Salvo em:", out)

## Próximos passos

- **Notebook 02:** extração completa (851 gravações) — MFCC + delta + delta2 + spectral, salvo em `data/processed/features.csv`
- **Notebook 03:** comparação de **XGBoost vs LDA vs Reg. Logística vs KNN** com split paciente-estratificado. Matriz de confusão a nível de paciente (agregação por voto majoritário) com contagem de pacientes.